### DAS Processing Demo

In [ ]:
# Import necessary dependencies
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import detrend

sys.path.append('..')

from src.utils import fk_transform, nextpow2, convert_to_tensor
from src.ani import bandpass_filter_tukey, temporal_normalization, spectral_whitening

In [ ]:
# Define paths
das_path = os.path.join('..', 'data', 'preprocessed', '20210927', '20210927_000000.npz')
ncf_path = os.path.join('..', 'data', 'ncf_raw', '20210927_000000_cc_080.npy')

#### 1. DAS Visualization

In [ ]:
# Load das data
das_data = np.load(das_path)

# Inspect variables inside 
for key in das_data:
    print(f'{key}')

In [ ]:
# DAS interrogator parameters
dt = 0.004 # Sampling rate in sec (or 250 Hz)
gauge_len = 16.0
chann_len = 8.16
dx = chann_len

In [ ]:
# Extract das data (channels, time)
das_array = das_data['data']
dt = das_data['dt']                 # Sampling rate in sec
N = das_data['data'].shape[1]       # Number of samples
T = N * dt                      # Total duration

print('DAS array shape:', das_array.shape)
print('Time axis shape:', das_data['t_axis'].shape)
print('Channel positions:', das_data['x_axis'].shape)
print(f'Sampling rates: {dt} s')
print(f'Total duration: {T/60:.2f} minutes')

In [ ]:
# Plot single channel
ch_idx = 0
plt.figure(figsize=(12, 4))
plt.plot(das_data['t_axis'], das_data['data'][ch_idx])
plt.xlabel('Time [s]')
plt.ylabel('Strain Rate (1/s)')
plt.title(f'Channel {das_data['x_axis'][ch_idx]}')
plt.xlim(das_data['t_axis'].min(), das_data['t_axis'].max())
plt.show()

In [ ]:
# Compute clipping value to avoid extreme outliers in the colormap
pclip = 99
das_clip = np.percentile(das_data['data'], pclip)

# Plot all channels as a heatmap
plt.figure(figsize=(12, 6))
plt.imshow(das_data['data'], aspect='auto',
           extent=[das_data['t_axis'][0], das_data['t_axis'][-1],
                   das_data['x_axis'][-1], das_data['x_axis'][0]],
                   vmin=-das_clip, vmax=das_clip,
           cmap='seismic')
plt.colorbar(label='Strain Rate (1/s)')
plt.xlabel('Time [s]')
plt.ylabel('Channel position')
plt.title('DAS Fiber Data')
plt.show()

#### 2. DAS Processing

In [ ]:
# Set preprocessing parameters
fs              = 250        # sampling frequency (Hz)
f1, f2          = 0.5, 10     # bandpass filter corners
Decimation      = 1          # if not 1, decimation factor after filtering
diff = False                 # whether to differentiate (strain → strain rate)
ram_win = 1                  # if 0, one-bit; otherwise temporal normalization windowm, usually  1/f1/5 ~ 1/f1/2 #
min_length = 60              # length of the segment in preprocessing, in sec, if shorter than this length, skip the file

In [ ]:
# For visualization purpose; select first 60-second window
samples_test = int(1*60.0 / dt)

if das_array.shape[1] < samples_test:
    samples_test = das_array.shape[1]

x_raw = das_array[:, :samples_test].astype(np.float32)
m, n = x_raw.shape
print(f'Matrix shape: {x_raw.shape}')

##### 2.1 Detrend

**What**
- Removing the mean (or linear trend) from each time-series so that long-period drifts or offsets do not bias subsequent filtering or correlation.

**Why**
- Filter design (especially bandpass) works better if the baseline is ~zero.
- Helps avoid edge‐effects in FFT or convolution.

In [ ]:
x_detrended = detrend(x_raw, axis=-1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
(ax1, ax2) = axes.flatten()

# Time axis length in seconds
t_max = samples_test * dt

# Channel positions: if x_axis goes from small to large, use direct order
channel_pos = das_data['x_axis']

# 1. Raw DAS
im1 = ax1.imshow(
    x_raw, aspect='auto',
    extent=[0, t_max, channel_pos[-1], channel_pos[0]],
    vmin=-das_clip, vmax=das_clip, cmap='seismic'
)
ax1.set_title("Raw DAS")
ax1.set_xlabel("Time [s]")
ax1.set_ylabel("Channel position")
fig.colorbar(im1, ax=ax1, label="Strain Rate (1/s)")

# 2. Detrended DAS
im2 = ax2.imshow(
    x_detrended, aspect='auto',
    extent=[0, t_max, channel_pos[-1], channel_pos[0]],
    vmin=-das_clip, vmax=das_clip, cmap='seismic'
)
ax2.set_title("Detrended DAS")
ax2.set_xlabel("Time [s]")
fig.colorbar(im2, ax=ax2, label="Strain Rate (1/s)")

plt.tight_layout()
plt.show()

##### 2.2 Tukey-windowed bandpass filter


**What**
1. Tukey window: 
    - A Tukey window is basically a rectangular window with cosine‐tapered ends. For $0 \le \alpha \le 1$, 

$$
w[n]=
\begin{cases}
\frac12\left[1-\cos\left(\frac{2\pi n}{\alpha N}\right)\right], & 0 \le n < \frac{\alpha N}{2},\\
1, & \frac{\alpha N}{2} \le n \le N-\frac{\alpha N}{2},\\
w[N-n], & \text{otherwise (symmetry)}
\end{cases}
$$

2. Band-pass filter + taper
    - We apply tapering to reduce spectral leakage (especially end‐effects) then apply a Butterworth band‐pass with normalized cut‐offs:
    $$\text{low} = \frac{f_1}{f_{\text{Nyq}}}, \; \text{high} = \frac{f_2}{f_{\text{Nyq}}}$$

**Why**
- The bandpass isolates the frequency band of interest.
- The Tukey taper prevents abrupt edges (which in Fourier domain introduce sidelobes).

In [ ]:
x_filtered = bandpass_filter_tukey(x_detrended, fs, f1, f2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
(ax1, ax2) = axes.flatten()

# 1. Raw DAS
im1 = ax1.imshow(
    x_raw, aspect='auto',
    extent=[0, t_max, channel_pos[-1], channel_pos[0]],
    vmin=-das_clip, vmax=das_clip, cmap='seismic'
)
ax1.set_title("Raw DAS")
ax1.set_xlabel("Time [s]")
ax1.set_ylabel("Channel position")
fig.colorbar(im1, ax=ax1, label="Strain Rate (1/s)")

# 2. Filtered DAS
im2 = ax2.imshow(
    x_filtered, aspect='auto',
    extent=[0, t_max, channel_pos[-1], channel_pos[0]],
    vmin=-das_clip, vmax=das_clip, cmap='seismic'
)
ax2.set_title("Filtered DAS")
ax2.set_xlabel("Time [s]")
fig.colorbar(im2, ax=ax2, label="Strain Rate (1/s)")

plt.tight_layout()
plt.show()

##### 2.3 Decimation

**What**
- Down-sampling the data by a factor Decimation.
- If original sampling rate is $f_s$​, and you decimate by factor $D$, then new sampling rate:
$$f_\text{new} = \frac{f_s}{D}$$

**Why**
- Reduces data size / computational cost (especially for very high sample rates).

In [ ]:
x_deci = x_filtered[:, ::Decimation]    # keep every 5th sample in time for each channel
fs_deci = fs / Decimation               # the new effective sampling rate after decimation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
(ax1, ax2) = axes.flatten()

# 1. Raw DAS
im1 = ax1.imshow(
    x_raw, aspect='auto',
    extent=[0, t_max, channel_pos[-1], channel_pos[0]],
    vmin=-das_clip, vmax=das_clip, cmap='seismic'
)
ax1.set_title("Raw DAS")
ax1.set_xlabel("Time [s]")
ax1.set_ylabel("Channel position")
fig.colorbar(im1, ax=ax1, label="Strain Rate (1/s)")

# 2. Decimated-Filtered DAS
im2 = ax2.imshow(
    x_deci, aspect='auto',
    extent=[0, t_max, channel_pos[-1], channel_pos[0]],
    vmin=-das_clip, vmax=das_clip, cmap='seismic'
)
ax2.set_title("Decimated-Filtered DAS")
ax2.set_xlabel("Time [s]")
fig.colorbar(im2, ax=ax2, label="Strain Rate (1/s)")

plt.tight_layout()
plt.show()

##### 2.4 Common mode noise

**What**
- Common‐mode noise refers to signals or disturbances that appear simultaneously and in‐phase on all or many channels of the DAS array.
- Removes signals that are coherent across channels (e.g., cable motion, interrogator noise).

**Why**
- Subtracting the median from each channel yields a residual signal where each channel has had the shared noise removed, leaving mostly the channel‐specific fluctuations.

In [ ]:
x_med = x_deci - np.median(x_deci, 0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
(ax1, ax2) = axes.flatten()

# 1. Raw DAS
im1 = ax1.imshow(
    x_raw, aspect='auto',
    extent=[0, t_max, channel_pos[-1], channel_pos[0]],
    vmin=-das_clip, vmax=das_clip, cmap='seismic'
)
ax1.set_title("Raw DAS")
ax1.set_xlabel("Time [s]")
ax1.set_ylabel("Channel position")
fig.colorbar(im1, ax=ax1, label="Strain Rate (1/s)")

# 2. Remove median (common mode noise)
im2 = ax2.imshow(
    x_med, aspect='auto',
    extent=[0, t_max, channel_pos[-1], channel_pos[0]],
    vmin=-das_clip, vmax=das_clip, cmap='seismic'
)
ax2.set_title("Median-removed DAS")
ax2.set_xlabel("Time [s]")
fig.colorbar(im2, ax=ax2, label="Strain Rate (1/s)")

plt.tight_layout()
plt.show()

##### 2.5 Temporal normalization

**What**
- Normalization in time used to reduce the influence of transient large‐amplitude events and non-stationary noise sources.

1. One-bit normalization (if ``window_time == 0``)
    - This removes amplitude information, only preserving sign.
       
$$
x'_{c,t} = \mathrm{sign}(x_{c,t})
= 
\begin{cases}
+1, & x_{c,t} > 0,\\
-1, & x_{c,t} < 0,\\
0,  & x_{c,t} = 0.
\end{cases}
$$

2. Running absolute mean (RAM) normalization (if ``window_time > 0``)
    - Computing a running mean of the absolute values and divide each sample.
    - This ensures local amplitude normalization. 

$$
\mathrm{RAM}_{c,t}
= \frac{1}{n_{\rm win}}
\sum_{k = t-\frac{n_{\rm win}-1}{2}}^{\,t + \frac{n_{\rm win}-1}{2}}
\bigl|\,x_{c,k}\bigr|
\quad\text{and}\quad
x'_{c,t} = \frac{x_{c,t}}{\mathrm{RAM}_{c,t}}.
$$

**Why**
- One-bit removes amplitude information but emphasises phase/coherence.
- RAM preserves some amplitude info while suppressing large transients (e.g., earthquakes) that could dominate the stack.
- Both methods improve the convergence of cross-correlations towards ambient noise Green’s functions.

In [ ]:
x_tem_norm = temporal_normalization(x_med, fs_deci, ram_win)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
(ax1, ax2) = axes.flatten()

# 1. Raw DAS
im1 = ax1.imshow(
    x_raw, aspect='auto',
    extent=[0, t_max, channel_pos[-1], channel_pos[0]],
    vmin=-das_clip, vmax=das_clip, cmap='seismic'
)
ax1.set_title("Raw DAS")
ax1.set_xlabel("Time [s]")
ax1.set_ylabel("Channel position")
fig.colorbar(im1, ax=ax1, label="Strain Rate (1/s)")

# 2. (Temporal) Normalized DAS
im2 = ax2.imshow(
    x_tem_norm, aspect='auto',
    extent=[0, t_max, channel_pos[-1], channel_pos[0]],
    vmin=-das_clip, vmax=das_clip, cmap='seismic'
)
ax2.set_title("(Temporal) Normalized DAS")
ax2.set_xlabel("Time [s]")
fig.colorbar(im2, ax=ax2, label="Strain Rate (1/s)")

plt.tight_layout()
plt.show()

#### 3. Cross-Correlation

##### 3.1 Cross-Correlation

**What**

Cross‐correlation takes two signals (recorded at two channels/locations) and computes how similar they are as a function of time‐lag ($\tau$) In ambient noise interferometry, we exploit the idea that by cross‐correlating long records of ambient seismic (or DAS) noise between two receivers $i$ and $j$, we can approximate the Green’s function (impulse response) between those two points.
$$
\mathrm{CC}_{ij}(\tau)
= \frac{1}{T}
\int_{0}^{T} x_i(t + \tau)\;x_j(t)\;dt
$$

where $x_i(t)$ and $x_j(t)$ are the time-series at channels $i$ and $j$, and $T$ is the integration time.

**Why**
- Ambient noise fields are assumed diffuse and provide energy from many directions; the cross‐correlation builds up coherent arrivals corresponding to wave propagation between stations.

- By stacking many short segments (or long records), the noise “averages out” and the coherent arrival emerges (often visible as a v-shaped move-out vs offset in DAS arrays).

In [ ]:
# Set ambient noise cross-correlation parameters
is_spectral_whitening   = True
window_freq             = 0.0                   # 0 → aggressive whitening, else running mean in Hz
max_lag                 = 1.0                   # time lag in seconds
xcorr_seg               = 2.0                   # segment length in seconds for CC window
npts_lag                = int(max_lag * fs_deci)               
npts_seg                = int(xcorr_seg * fs_deci)

**Key parameters**
- `max_lag = 1.0` (seconds)         -- compute up to ± 1s lag (time shift) from zero-lag for each channel pair
- `xcorr_seg = 2.0` (seconds)       -- each segment used in CC is 2s long
- `npts_lag` and `npts_seg`         -- convert seconds to sample-points based on `fs_deci`

**Explanation**
- We take the processed DAS data and divide it into consecutive windows of length `xcorr_seg` seconds. 
- Then for each window, we compute a cross-correlation and stack (average) all these window-by-window cross-correlations.

**Benefits**
1. Enables stacking many short‐duration cross‐correlations → improved SNR.
2. Keeps computational burden manageable.
3. Assumes approximate stationarity of noise within each window, which is more realistic than assuming stationarity for an entire long record.

##### 3.2 Spectral Whitening

**What**

- Many ambient noise records (and their cross‐spectra) are dominated by large amplitude variations in some frequency bands (e.g., microseisms) and weak in others. Spectral whitening aims to flatten the amplitude spectrum across frequencies within the band of interest — basically making the spectrum more “white” — while preserving phase information which encodes propagation paths. 

1. Aggressive whitening (when `window_freq == 0`)
    - Sets amplitude to unity across all frequency bins within band, retaining only phase.

$$
\tilde X(\omega) = \frac{X(\omega)}{|X(\omega)|}
= e^{\,i\,\phi(\omega)}
$$

2. Running-mean whitening (when `window_freq > 0`)
    - Smooths the amplitude spectrum using a running window of width `window_freq` (Hz), then divides the original amplitude by that smoothed amplitude to flatten spectrum gradually rather than completely.

$$
\tilde X(\omega)
= e^{\,i\,\phi(\omega)} \;\cdot\; \frac{|X(\omega)|}
{|X(\omega)|_{\text{window}}}
$$

**Why**
- Suppresses strong persistent noise features (e.g., tonal noise, microseism peaks) that might bias correlation results or lead to false coherence.
- Improves retrieval of weaker but distributed energy sources and enhances signal‐to‐noise of the cross‐correlation.

In [ ]:
# Convert to tensor 
x_raw_torch = convert_to_tensor(x_raw)                          # (nch × nt)
nch, nt = x_raw_torch.shape
device = x_raw_torch.device

# Determine FFT length using nextpow2
fft_len = int(nextpow2(torch.tensor([nt], device=device))[0].item())
df = fs_deci / fft_len                                          # frequency bin spacing 

print(f'FFT length for whitening: {fft_len}, df={df:.4f} Hz')

# Forward FFT (time axis)
fft_raw = torch.fft.rfft(x_raw_torch, n=fft_len, dim=-1)        # (nch × nfreq)

# Apply spectral whitening
fft_white = spectral_whitening(fft_raw, df=df, window_freq=window_freq, f1=f1, f2=f2)

# Inverse FFT → x_whitened (time domain)
x_white_torch = torch.fft.irfft(fft_white, n=fft_len, dim=-1)
x_whitened = x_white_torch[:, :nt].cpu().numpy()

# Compute f-k transform for raw and whitened
freqs_raw, k_raw, fk_raw = fk_transform(x_raw, float(dt), dx)
freqs_wh, k_wh, fk_wh = fk_transform(x_whitened, float(dt), dx)

fk_raw_amp = torch.abs(fk_raw).cpu().numpy()
fk_wh_amp = torch.abs(fk_wh).cpu().numpy()

In [ ]:
# Compute log10 amplitude
eps = 1e-6  # small stabilizer for log

fk_raw_log = np.log10(fk_raw_amp + eps)
fk_wh_log  = np.log10(fk_wh_amp  + eps)

# Percentile-based clipping (symmetric across raw/whitened)
vmin_raw, vmax_raw = np.percentile(fk_raw_log,  [1, 99])
vmin_wh,  vmax_wh  = np.percentile(fk_wh_log,   [1, 99])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

(ax1, ax2), (ax3, ax4) = axes

fk_clip_raw = np.percentile(fk_raw_amp, 99)
fk_clip_wh  = np.percentile(fk_wh_amp, 99)

# 1. x_raw (time-space)
im1 = ax1.imshow(
    x_raw, aspect='auto',
    extent=[0, T, channel_pos[-1], channel_pos[0]],
    vmin=-das_clip, vmax=das_clip,
    cmap='seismic'
)
ax1.set_title('Raw DAS (x–t)')
ax1.set_xlabel('Time [s]')
ax1.set_ylabel('Channel position')
fig.colorbar(im1, ax=ax1)

# 2. x_whitened (time-space)
im2 = ax2.imshow(
    x_whitened, aspect='auto',
    extent=[0, T, channel_pos[-1], channel_pos[0]],
    vmin=-das_clip, vmax=das_clip,
    cmap='seismic'
)
ax2.set_title('Whitened DAS (x–t)')
ax2.set_xlabel('Time [s]')
fig.colorbar(im2, ax=ax2)

# 3. Raw f–k amplitude
im3 = ax3.imshow(
    fk_raw_log,
    aspect='auto',
    extent=[freqs_raw[0].item(), freqs_raw[-1].item(),
            k_raw[-1].item(),   k_raw[0].item()],
    vmin=vmin_raw, vmax=vmax_raw,
    cmap='inferno'
)
ax3.set_title('Raw DAS – f–k spectrum (log amplitude)')
ax3.set_xlabel('Frequency [Hz]')
ax3.set_ylabel('Wavenumber [cycles/m]')
fig.colorbar(im3, ax=ax3, label='log10 amplitude')

# 4. Whitened f–k amplitude
im4 = ax4.imshow(
    fk_wh_log,
    aspect='auto',
    extent=[freqs_wh[0].item(), freqs_wh[-1].item(),
            k_wh[-1].item(),   k_wh[0].item()],
    vmin=vmin_wh, vmax=vmax_wh,
    cmap='inferno'
)
ax4.set_title('Whitened DAS – f–k spectrum (log amplitude)')
ax4.set_xlabel('Frequency [Hz]')
fig.colorbar(im4, ax=ax4, label='log10 amplitude')

plt.tight_layout()
plt.show()

##### 3.3 Equation summary
1. Forward FFT
$$
X_i(\omega)
= \mathcal{F}\{\,x_i(t)\}
$$

2. Whitening
$$
\tilde X_i(\omega) =
\begin{cases}
e^{\,i\,\angle X_i(\omega)}, & \text{if } \text{window}_f = 0, \\[1em]
e^{\,i\,\angle X_i(\omega)} \;\cdot\; \dfrac{\bigl|X_i(\omega)\bigr|}{\text{smoothed}\!\bigl(\,|X_i(\omega)|\bigr)}, & \text{if } \text{window}_f > 0.
\end{cases}
$$

3. Cross-spectrum between two channels
$$
C_{ij}(\omega) = \tilde X_i^*(\omega)\;\tilde X_j(\omega)
$$

4. Inverse FFT back to time-lag domain
$$
\mathrm{CC}_{ij}(\tau) = \mathcal{F}^{-1}\{\,C_{ij}(\omega)\}
$$

#### 4. Cross-correlation Code example

You can run the following command line at the root directory:
```bash
python -m src.cc --data_root ./data/preprocessed --output_root ./data/ncf_raw --njobs 4 --use_gpu --verbose
```

#### 5. NCF Visualization

In [ ]:
# Load ncf data
ncf_data = np.load(ncf_path)

# Extract metadata
n_receivers, n_lags = ncf_data.shape

In [ ]:
# Max lag and sample rate (from settings)
max_lag = 1.0               # seconds (as set)
npts_lag = int(max_lag * fs)

In [ ]:
# Lag axis spans from –max_lag … +max_lag
lag_axis = np.linspace(-max_lag, +max_lag, n_lags)

In [ ]:
# Extract ncf data
print(f'Receiver channels: {n_receivers}')
print(f'Lag dimension: {n_lags} samples → {2 * max_lag:.3f} s total window')
print(f'Processed sampling rate: {fs:.1f} Hz → dt_proc = {dt:.4f} s')
print(f'Lag axis (first/last): {lag_axis[0]:.3f} s, {lag_axis[-1]:.3f} s')
print('NCF data shape:', ncf_data.shape)

In [ ]:
# Cable configuration
first_chan = 399
last_chan = 748
dx = chann_len
channels = np.arange(first_chan, last_chan + 1)

if len(channels) != n_receivers:
    channels = np.arange(n_receivers)

distance_axis = (channels - channels[0]) * dx

In [ ]:
# Compute clipping value to avoid extreme outliers in the colormap
pclip = 99
ncf_clip = np.percentile(ncf_data, pclip)

In [ ]:
# Plot NCF
plt.figure(figsize=(10, 6))

img = plt.imshow(ncf_data,
                 extent=[lag_axis[0], lag_axis[-1], distance_axis[-1], distance_axis[0]],
                 aspect='auto',
                 cmap='seismic', 
                 vmin=-ncf_clip, vmax=ncf_clip, 
                 interpolation='nearest')

plt.colorbar(img, label='Correlation amplitude')
plt.xlabel('Lag time (s)')
plt.ylabel('Distance along array (m)')
plt.title(f'NCF: {os.path.basename(ncf_path)}')
plt.tight_layout()
plt.show()

#### 6. Data format discussion

##### 6.1 Data dimensions
- DAS array shape: (350, 157500):
    - 350 = number of channels (spatial)
    - 157500 = number of samples (time) for the entire record
    - dt=0.004s → sampling rate = 250 Hz
    - 157500 samples × 0.004 s/sample = 630 s = 10.50 minutes

- NCF file shape: (350, 501)
    - 350 = number of “receivers” (channels) being correlated against a virtual source channel
    - 501 = number of lag samples in the cross-correlation output
    - Processed sampling rate = 250 Hz → df_proc = 0.004 s (`Decimation =1`)
    - 501 samples × 0.004 s ≈ 2.0 s total lag window
    - Lag axis spans from –1.0 s to +1.0 s (which is symmetric about zero‐lag)

##### 6.2 NCF Interpretation
- Rows (350): each row corresponds to one “receiver” channel in the array (or rather one channel index) being correlated with a fixed virtual source channel.
- Columns (501): each column corresponds to a lag index $\tau$ (in seconds). With 501 points, we have lags from –1.0 s to +1.0 s (zero‐lag at column ~251). The amplitude at (row, column) gives the correlation coefficient (or correlation amplitude) at lag $\tau$ between source channel and receiver channel.

##### 6.3 Spatial axis interpretation
- One axis is offset or distance: since receivers are spaced uniformly (channel spacing dx = 8.16 m), we can convert channel index difference into offset. So each row also corresponds to an offset distance from the source.
- The other axis is lag time: correlation between source and receiver yields arrival peaks at positive or negative lag according to whether a wave travels from source → receiver or receiver → source (depending how your code defines negative vs positive).

##### 6.4 Virtual Shot Gather (VSG)
- In passive ambient noise DAS workflow, we treat one channel as a virtual source, by correlating it with all other channels (or the full array). - This yields many receiver traces for that “source” — forming a gather as if we had actively placed a source at that channel and recorded with the array.
- Such a gather shows wavefronts (arrival times across offsets) emerging symmetrically (± lag) if sources and wave‐fields are well‐distributed.